###  Counts and prints the number of unique employee_id values in the targets and salesperson tables.

In [0]:
t = spark.table("sales_lakehouse.silver.targets")
s = spark.table("sales_lakehouse.silver.salesperson")

print("targets distinct employee_id:", t.select("employee_id").distinct().count())
print("salesperson distinct employee_id:", s.select("employee_id").distinct().count())

### Finds employee_id values that exist in targets but not in salesperson, then prints how many are missing

In [0]:
missing = (t.select("employee_id").distinct()\
    .join(s.select("employee_id").distinct(), on="employee_id", how="left_anti"))
print("missing employee_id (targets not in salesperson):", missing.count())
display(missing.limit(100))

### Creates the fact_targets table by linking each target record to the correct salesperson key and month, then saves it as a Delta table

In [0]:
# =========================
# GOLD NOTEBOOK: fact_targets (final)
# =========================

from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS sales_lakehouse.gold")

targets = spark.table("sales_lakehouse.silver.targets")  # employee_id, target, target_month
sp_map  = (spark.table("sales_lakehouse.silver.salesperson")
           .select("employee_id", "employee_key")
           .dropDuplicates())

fact_targets = (
    targets
    .withColumn("date_key", F.date_format(F.col("target_month"), "yyyyMMdd").cast("int"))
    .join(sp_map, on="employee_id", how="inner")  # mapping employee_id -> employee_key
    .select(
        "employee_key",
        "date_key",
        F.col("target").cast("decimal(10,2)").alias("target_sales_amount")
    )
)

(fact_targets.write
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .format("delta")
 .saveAsTable("sales_lakehouse.gold.fact_targets"))

print("Wrote table: sales_lakehouse.gold.fact_targets")
display(spark.table("sales_lakehouse.gold.fact_targets").limit(20))

In [0]:
ft = spark.table("sales_lakehouse.gold.fact_targets")
dd = spark.table("sales_lakehouse.gold.dim_date").select("date_key").distinct()
ft.join(dd, "date_key", "left_anti").count()